# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
rel = "hf://datasets/FlyRank/internship-warehouse"

In [12]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

,count_star()
0,11694072


In [13]:
con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [14]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis

One row represents one search query for one landing page on one day.

## Time window

This notebook uses data from March 2026 (month = "2026-03") to avoid using the final outcome month (June 2026).

In [18]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-13,client_e00b29e582949543,content_e3491394a9f3e2b3,2
1,2026-06-13,client_e00b29e582949543,content_83b20e2cd4d5437f,2
2,2026-06-13,client_e00b29e582949543,content_04307a138472fc03,2
3,2026-06-13,client_e00b29e582949543,content_638ac5ad8023d177,2
4,2026-06-13,client_e00b29e582949543,content_ff881e76e85001f4,2


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- clicks
- impressions
- ctr
- position
- device

### Label
- future_clicks

### Context
- date
- page
- query

### Excluded
- June 2026 because it is reserved as the final evaluation period.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt
0,2026-06-13,client_e00b29e582949543,content_e3491394a9f3e2b3,2
1,2026-06-13,client_e00b29e582949543,content_83b20e2cd4d5437f,2
2,2026-06-13,client_e00b29e582949543,content_04307a138472fc03,2
3,2026-06-13,client_e00b29e582949543,content_638ac5ad8023d177,2
4,2026-06-13,client_e00b29e582949543,content_ff881e76e85001f4,2


In [16]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

,total_rows,start_date,end_date
0,11694072,2026-06-01,2026-06-30


In [17]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,554216


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

- Only one month is analyzed.
- Historical search data may be incomplete.
- The dataset does not include user-level behavior.
- Results should be used for decision support, not as proof of causation.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [8]:
!pip install -q duckdb huggingface_hub pandas pyarrow

In [4]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print(HF_TOKEN is not None)

True


In [9]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [10]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")